# ♟️ Chess Numba — SF19 Lazy SMP 新舊多執行緒對戰 (Colab Tournament Runner)

本 Notebook 用於在 Google Colab 環境下，直接比較 **此版本多執行緒 (`chess_engine.classical`, SF19 對齊版)** 與 **上個版本多執行緒 (`chess_engine.classical_old`, 舊版基準)** 在**固定每步時間**下的實戰棋力差距：

### 🎯 對戰雙方差異
* **Engine 1 (`New_SF19SMP`, `chess_engine.classical`)**：
  * 取消 Helper 深度跳躍，以 `ASPIRATION_WINDOW_BASE + (thread_idx & 7)` 錯開視窗。
  * 保留中斷時的當層根節點候選棋步、PV 與 `inexact_lower` / `inexact_upper` 邊界，並加入 SF19 `forgottenMate` / `abortedLossSearch` 回滾保護。
  * 以 SF19 `score - min_score + 14` 分數加權投票彙整 Thread 0 與全部 Helpers，優先已證實的最短殺棋、排除未證實敗著，並於 Helper 勝出時補印 UCI `info`。
* **Engine 2 (`Old_SMP`, `chess_engine.classical_old`)**：
  * 舊版 Lazy SMP（2/3 Helpers 在深度 >= 4 跳過偶/奇深度、中斷即丟棄當層、無投票且必須同時滿足 `h_depth > last_completed_depth` 與 `h_score > last_score` 才採用 Helper）。

### ⚠️ 開跑前重要提醒
1. **同步最新程式碼**：Colab 預設會從 GitHub 拉取 `BRANCH`；若你本機修改尚未 `git push`，請先執行 `git push`，或將整個專案打包為 `.zip` 上傳到 Google Drive 並填入下方 `DRIVE_REPO_ZIP`。
2. **為何必須用「固定時間 (`--time`)」而非「固定節點 (`--nodes`)」**：固定節點會把總節點數平分給多執行緒，無法測出平行搜尋的時間加速與投票收益。
3. **避免 CPU 超額搶核**：多執行緒時務必控制 `CONCURRENCY * max(THREADS1, THREADS2) <= CPU 核心數`，否則核心排程競爭會扭曲限時結果。


In [ ]:
# =====================================================================
# 步驟 1: 檢視 Colab 雲端硬體規格與 CPU 核心數
# =====================================================================
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("os.cpu_count():", os.cpu_count())

for cmd in (
    ["bash", "-lc", "lscpu | grep -E 'Model name|Socket|Thread|Core|NUMA|CPU\(s\)' || true"],
    ["free", "-h"],
):
    if shutil.which(cmd[0]):
        res = subprocess.run(cmd, text=True, capture_output=True)
        if res.stdout.strip():
            print(res.stdout.strip())


In [ ]:
# =====================================================================
# 步驟 2: 設定對戰參數與 Google Drive 持久化路徑
# =====================================================================
import os
from pathlib import Path

# --- 程式碼來源設定 ---
REPO_URL = "https://github.com/b08203031/chess_numba.git"
BRANCH = "main"
# 選填：若尚未 git push，可將本機 repo 壓成 zip 上傳 Google Drive，例如：
# DRIVE_REPO_ZIP = "/content/drive/MyDrive/chess_numba_sync.zip"
DRIVE_REPO_ZIP = ""

# --- 引擎與執行緒設定 ---
RUN_TAG = "sf19_smp_vs_old_smp_t4_250ms"
NAME1 = "New_SF19SMP"
NAME2 = "Old_SMP"
ENGINE1_MODULE = "chess_engine.classical"
ENGINE2_MODULE = "chess_engine.classical_old"

THREADS = 4                  # 建議 <= 實體核心數（Colab 8 vCPU 實為 4 實體核心 x 2 HT，建議設 4）
THREADS1 = THREADS
THREADS2 = THREADS

# --- 固定時間對戰規格 ---
MOVETIME_MS = 500            # 每步固定時間 (ms)；250ms x 400局(c=1) 約 1.5~2.5 小時，可視需求調成 150~500
GAMES = 400                  # 必須為偶數（成對換先）
TT_MB = 128                  # 每個對局 context 的置換表大小 (MB)
CONCURRENCY = 0              # 0 = 自動依 CPU 核心數計算 max(1, (cpu_count - 1) // max(THREADS1, THREADS2))

# --- 開局、SPRT 與裁決設定 ---
OPENING_MOVES = 8
SEED = 0                     # 固定隨機種子便於復現；想換開局順序可設 time.time_ns()
USE_SPRT = True
SPRT_ELO0 = 0.0
SPRT_ELO1 = 10.0
ADJUDICATE = True

# --- Google Drive 同步設定 ---
USE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/chess_numba_data/smp_tournaments")
LOCAL_ROOT = Path("/content/smp_tournament_runs")

cpu_count = os.cpu_count() or 2
max_side_threads = max(THREADS1, THREADS2)
if CONCURRENCY <= 0:
    ACTUAL_CONCURRENCY = max(1, (cpu_count - 1) // max_side_threads)
else:
    ACTUAL_CONCURRENCY = CONCURRENCY

DRIVE_RUN_DIR = DRIVE_ROOT / RUN_TAG
LOCAL_RUN_DIR = LOCAL_ROOT / RUN_TAG

print(f"RUN_TAG={RUN_TAG} | {NAME1}(T={THREADS1}) vs {NAME2}(T={THREADS2})")
print(f"games={GAMES}, movetime={MOVETIME_MS}ms, TT={TT_MB}MB, cpu_count={cpu_count}, concurrency={ACTUAL_CONCURRENCY}")
if ACTUAL_CONCURRENCY * max_side_threads > cpu_count:
    print(
        f"[警告] concurrency * max_threads ({ACTUAL_CONCURRENCY * max_side_threads}) "
        f"超過 CPU(s) ({cpu_count})，可能造成排程競爭與時間波動！"
    )
elif cpu_count < max_side_threads:
    print(
        f"[警告] 目前僅有 {cpu_count} 顆 CPU，設定 Threads={max_side_threads} 可能無法獲得線性擴展；"
        "建議將 THREADS 調降為 <= cpu_count。"
    )


In [ ]:
# =====================================================================
# 步驟 3: 掛載 Google Drive、取得專案程式碼並安裝相依套件
# =====================================================================
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = ("google.colab" in sys.modules) or (os.name != "nt" and Path("/content").is_dir())
WORKDIR = Path("/content/chess_numba") if IN_COLAB else Path.cwd()

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_COLAB:
    if DRIVE_REPO_ZIP and Path(DRIVE_REPO_ZIP).is_file():
        print(f"[Repo] Extracting {DRIVE_REPO_ZIP} -> {WORKDIR}")
        if WORKDIR.exists():
            shutil.rmtree(WORKDIR)
        WORKDIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DRIVE_REPO_ZIP, "r") as zf:
            zf.extractall(WORKDIR)
        # 若 zip 內含單一層目錄（例如 chess_numba/tools/tournament.py），自動攤平到 WORKDIR
        if not (WORKDIR / "tools" / "tournament.py").is_file():
            subdirs = [p for p in WORKDIR.iterdir() if p.is_dir()]
            if len(subdirs) == 1 and (subdirs[0] / "tools" / "tournament.py").is_file():
                inner = subdirs[0]
                for item in list(inner.iterdir()):
                    shutil.move(str(item), str(WORKDIR / item.name))
                inner.rmdir()
    else:
        if not (WORKDIR / ".git").is_dir():
            print(f"[Git] Cloning {REPO_URL} (branch={BRANCH}) -> {WORKDIR}")
            subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
        else:
            print(f"[Git] Updating {WORKDIR} from origin/{BRANCH}")
            subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", BRANCH], check=True)
            subprocess.run(["git", "-C", str(WORKDIR), "checkout", "-f", BRANCH], check=True)
            subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
            subprocess.run(["git", "-C", str(WORKDIR), "clean", "-fd"], check=True)

    os.chdir(WORKDIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "numba", "chess", "numpy", "matplotlib", "scipy"],
        check=True,
    )
else:
    DRIVE_RUN_DIR = WORKDIR / "tournament_analysis" / RUN_TAG
    LOCAL_RUN_DIR = DRIVE_RUN_DIR

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "log", "-1", "--oneline"], check=False)

# 防呆檢查：確認載入的程式碼已具備 --threads1 / SF19 select_best_thread
help_out = subprocess.run(
    [sys.executable, "tools/tournament.py", "--help"],
    cwd=WORKDIR,
    text=True,
    capture_output=True,
    check=True,
).stdout
if "--threads1" not in help_out:
    raise RuntimeError(
        "目前 WORKDIR 的 tools/tournament.py 尚未包含 --threads1！"
        "請先將本機修改 git push 至 GitHub，或將最新專案打包上傳並設定 DRIVE_REPO_ZIP。"
    )
search_src = (WORKDIR / "chess_engine" / "classical" / "search.py").read_text(encoding="utf-8")
if "def select_best_thread(" not in search_src:
    raise RuntimeError(
        "目前 WORKDIR 的 chess_engine/classical/search.py 尚未包含 SF19 select_best_thread！"
        "請先同步最新程式碼再執行對戰。"
    )
print("[OK] 已確認 WORKDIR 包含 SF19 Lazy SMP 與多執行緒 tournament 工具。")


In [ ]:
# =====================================================================
# 步驟 4: 核對新舊版本差異並執行前置單元測試（同時完成 Numba JIT 編譯/快取）
# =====================================================================
import subprocess
import sys

print("=" * 60)
print("🔍 classical vs classical_old 代碼差異摘要：")
print("=" * 60)
diff_res = subprocess.run(
    [sys.executable, "tools/sync_classical_old.py", "--diff-code"],
    cwd=WORKDIR,
    text=True,
    capture_output=True,
)
for line in diff_res.stdout.splitlines():
    if line.startswith("[!]") or line.startswith("[OK]") or line.startswith("---") or line.startswith("+++"):
        print(line)

print("\n" + "=" * 60)
print("🔥 執行 Lazy SMP / Tournament / 契約單元測試（首次冷啟動會編譯 Numba JIT，約需 2~5 分鐘）：")
print("=" * 60)
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "-v",
        "tests.test_lazy_smp",
        "tests.test_tournament_threads",
        "tests.test_search_core_contracts",
    ],
    cwd=WORKDIR,
    check=True,
)


In [ ]:
# =====================================================================
# 步驟 5: 啟動新舊多執行緒固定時間錦標賽（即時輸出並同步至 Google Drive）
# =====================================================================
import shutil
import subprocess
import sys

PGN_PATH = LOCAL_RUN_DIR / f"{RUN_TAG}.pgn"
LOG_PATH = LOCAL_RUN_DIR / f"{RUN_TAG}.log"

cmd = [
    sys.executable,
    "-u",
    "tools/tournament.py",
    "--backend", "inprocess",
    "--engine1-module", ENGINE1_MODULE,
    "--engine2-module", ENGINE2_MODULE,
    "--name1", NAME1,
    "--name2", NAME2,
    "--threads1", str(THREADS1),
    "--threads2", str(THREADS2),
    "--games", str(GAMES),
    "--time", str(MOVETIME_MS),
    "--concurrency", str(ACTUAL_CONCURRENCY),
    "--tt-mb", str(TT_MB),
    "--opening-moves", str(OPENING_MOVES),
    "--seed", str(SEED),
    "--pgn", str(PGN_PATH),
]

if USE_SPRT:
    cmd.extend(["--sprt-elo0", str(SPRT_ELO0), "--sprt-elo1", str(SPRT_ELO1)])
else:
    cmd.append("--no-sprt")

cmd.append("--adjudicate" if ADJUDICATE else "--no-adjudicate")

print("執行命令:")
print(" ".join(cmd))
print("=" * 60)

returncode = 1
try:
    with LOG_PATH.open("w", encoding="utf-8", newline="\n") as log_f:
        proc = subprocess.Popen(
            cmd,
            cwd=WORKDIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_f.write(line)
            if "After " in line:
                log_f.flush()
                if USE_DRIVE and PGN_PATH.exists() and LOCAL_RUN_DIR != DRIVE_RUN_DIR:
                    shutil.copy2(PGN_PATH, DRIVE_RUN_DIR / PGN_PATH.name)
        returncode = proc.wait()
finally:
    if USE_DRIVE and LOCAL_RUN_DIR != DRIVE_RUN_DIR:
        for artifact in (PGN_PATH, LOG_PATH):
            if artifact.exists():
                shutil.copy2(artifact, DRIVE_RUN_DIR / artifact.name)
        print(f"\n[Drive] 已同步棋譜與日誌至: {DRIVE_RUN_DIR}")

if returncode != 0:
    raise RuntimeError(f"tournament.py exited with code {returncode}")


In [ ]:
# =====================================================================
# 步驟 6: 產出對局統計報告與搜尋樹指標診斷
# =====================================================================
import shutil
import subprocess
import sys

ANALYSIS_DIR = LOCAL_RUN_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        "tournament_analysis/統計數據.py",
        "--pgn", str(PGN_PATH),
        "--target", NAME1,
        "--out-dir", str(ANALYSIS_DIR),
    ],
    cwd=WORKDIR,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "tournament_analysis/parse_search_stats.py",
        "--pgn", str(PGN_PATH),
        "--name1", NAME1,
        "--name2", NAME2,
        "--out-dir", str(ANALYSIS_DIR),
    ],
    cwd=WORKDIR,
    check=True,
)

for report_name in ("game_stats_report.txt", "search_stats_report.txt"):
    rpath = ANALYSIS_DIR / report_name
    if rpath.exists():
        print("\n" + "=" * 60)
        print(f"📊 {report_name}")
        print("=" * 60)
        print(rpath.read_text(encoding="utf-8"))

if USE_DRIVE and LOCAL_RUN_DIR != DRIVE_RUN_DIR:
    drive_analysis = DRIVE_RUN_DIR / "analysis"
    drive_analysis.mkdir(parents=True, exist_ok=True)
    for item in ANALYSIS_DIR.iterdir():
        if item.is_file():
            shutil.copy2(item, drive_analysis / item.name)
    print(f"[Drive] 已同步報告與圖表至: {drive_analysis}")


In [ ]:
# =====================================================================
# 步驟 7: 視覺化關鍵圖表展示
# =====================================================================
from IPython.display import Image, display

charts = [
    ("累積分數曲線 (Cumulative Score)", "cumulative_score.png"),
    ("Elo 置信區間收斂圖 (Elo CI)", "elo_ci.png"),
    ("每手搜尋深度 (Depth per Move)", "depth_per_move.png"),
    ("每手耗時 (Time per Move)", "time_per_move.png"),
    ("同深度節點比例分佈 (Same-Depth Node Ratio)", "same_depth_node_ratio.png"),
    ("平均節點數 vs 深度散佈 (Avg Nodes vs Depth)", "avg_nodes_vs_depth.png"),
    ("黑白勝負分佈 (WDL by Color)", "wdl_by_color.png"),
    ("搜尋深度 CDF (Depth CDF)", "depth_cdf.png"),
]

for title, filename in charts:
    path = ANALYSIS_DIR / filename
    if path.exists():
        print(f"\n📈 {title}：")
        display(Image(filename=str(path)))


In [ ]:
# =====================================================================
# 步驟 8: 打包所有報告、圖表、日誌與 PGN 棋譜並觸發下載
# =====================================================================
import datetime
import shutil
import zipfile

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = LOCAL_RUN_DIR / f"{RUN_TAG}_{timestamp}.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in (PGN_PATH, LOG_PATH):
        if item.exists():
            zf.write(item, arcname=item.name)
    if ANALYSIS_DIR.exists():
        for item in sorted(ANALYSIS_DIR.iterdir()):
            if item.is_file():
                zf.write(item, arcname=f"analysis/{item.name}")

if USE_DRIVE and LOCAL_RUN_DIR != DRIVE_RUN_DIR:
    shutil.copy2(zip_path, DRIVE_RUN_DIR / zip_path.name)
    print(f"[Drive] 已備份壓縮檔至: {DRIVE_RUN_DIR / zip_path.name}")

print(f"📦 已打包完成: {zip_path}")
try:
    from google.colab import files
    files.download(str(zip_path))
    print("⬇️ 已觸發下載！")
except ImportError:
    print(f"💡 本地環境執行，壓縮包位於: {zip_path}")


In [ ]:
# =====================================================================
# 步驟 9 (選用): 結束後釋放 Colab VM 資源（預設關閉，避免 Run All 誤斷線）
# =====================================================================
UNASSIGN_RUNTIME = False

if UNASSIGN_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
